In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier  # 분류 모델로 변경
from sklearn.metrics import accuracy_score, classification_report
from sklearn.inspection import permutation_importance

import joblib

In [ ]:
# 1. 데이터 로드 및 전처리 
df = pd.read_csv('dataset/heart_disease_uci_korean.csv', encoding='cp949')
df = df.drop(columns=['주요 혈관 수', '심장 혈류 상태']) # 결측치 너무 많은 컬럼 삭제

# 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

cols_to_fix = ['안정 시 혈압', '나이', '최대 심박수']
for col in cols_to_fix:
    df[col] = df[col].replace(0, np.nan)

In [ ]:
# 2. 결측치 채우기 (전체 데이터 대상)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())

object_cols = df.select_dtypes(include=['object']).columns
for col in object_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
# 2. Feature Engineering 
# 나이 대비 혈압 비율
df['나이 대비 혈압 비율'] = df['안정 시 혈압'] / df['나이']
# 고령자 여부 (60세 이상)
df['고령자 여부'] = (df['나이'] >= 60).astype(int)
# 최대 심박수 대비 현재 혈압 비율
df['최대 심박수 대비 현재 혈압 비율'] = df['최대 심박수'] / df['안정 시 혈압']
# 무한 값 처리.
df = df.replace([np.inf, -np.inf], np.nan).fillna(0)
# 타겟 변수 이진화 (0: 정상, 1: 질환)
df['target'] = (df['심장병 진단 결과'] > 0).astype(int)

In [ ]:
# 3. 히트맵 분석 (타겟은 target만 — 심장병 진단 결과 열은 상관행렬에서 제외)
plt.figure(figsize=(10, 8))
exclude_from_corr = {'심장병 진단 결과'}
numeric_cols = [c for c in df.select_dtypes(include=['number']).columns if c not in exclude_from_corr]
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('상관관계 히트맵')
plt.show()

In [ ]:
# 4. 라벨 인코딩 
categorical_cols = df.select_dtypes(include=['object']).columns
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

In [ ]:
# 5. Feature/Target X, y 분리
# 원본 타겟과 파생 타겟을 제외한 나머지를 X로
X = df.drop(['심장병 진단 결과', 'target'], axis=1)
y = df['target']

In [ ]:
# 6. 데이터 분할 (정확하게 정답 컬럼들만 제외)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# 7. 모델 학습 및 정확도 확인
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=3,
    random_state=42
)
model.fit(X_train, y_train)

In [ ]:
# 8. 예측 및 평가
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("\n===== 모델 성능 =====")
print(f"Accuracy: {accuracy:.4f}")
print(classification_report(y_test, y_pred))

In [ ]:
# 9. Feature Importance
importances = pd.Series(model.feature_importances_, index=X.columns)

plt.figure(figsize=(8, 5))
importances.sort_values().plot(kind='barh')
plt.title("Feature Importance (Heart Disease)")
plt.xlabel("Importance")
plt.show()

In [ ]:
# 10. Permutation Importance
perm = permutation_importance(
    model, X_test, y_test,
    n_repeats=10,
    random_state=42
)
perm_importance = pd.Series(perm.importances_mean, index=X.columns)

plt.figure(figsize=(8, 5))
perm_importance.sort_values().plot(kind='barh')
plt.title("Permutation Importance")
plt.xlabel("Importance")
plt.show()

In [ ]:
# 11. 모델 저장
joblib.dump(model, "model/heart_model.pkl")
joblib.dump(encoders, "model/heart_encoders.pkl")

In [ ]:
print("최종 검사 결과")
print(classification_report(y_test, y_pred))
print(f"정확도: {accuracy_score(y_test, y_pred):.4f}")
